<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.2-multi-agent/practice/GCP_Capstone_8.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 8.2 — Multi-Agent Orchestration

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: Install, Auth, and Shared Helpers

Run these three cells once. They install ADK, authenticate with Application Default Credentials (ADC), define the five DocuMind tools, and add a reusable `run_agent` helper used by every exercise below.

In [ ]:
!pip install -q google-adk google-genai
import os
from google.colab import auth
auth.authenticate_user()

os.environ['GOOGLE_CLOUD_PROJECT'] = 'documind-ai-YOUR-ID'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'  # global endpoint for Gemini 3.x generation
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print('ADK ready (Vertex + ADC)')

In [ ]:
from google.adk.tools import ToolContext

def search_documents(query: str) -> dict:
    """Search documents by keyword.
    Args:
        query: Search query.
    """
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report'}]}

def summarize_document(document_id: str, summary_type: str) -> dict:
    """Summarize a document.
    Args:
        document_id: Document ID.
        summary_type: brief/detailed/executive.
    """
    return {'summary': 'Summary of ' + document_id}

def extract_entities(text: str) -> dict:
    """Extract named entities from text.
    Args:
        text: Text to analyze.
    """
    return {'entities': {'people': ['John'], 'orgs': ['Acme']}}

def calculate_cost(page_count: int, tier: str) -> dict:
    """Calculate processing cost.
    Args:
        page_count: Pages.
        tier: standard/premium/enterprise.
    """
    rates = {'standard': 0.01, 'premium': 0.03, 'enterprise': 0.05}
    return {'cost_usd': round(page_count * rates.get(tier, 0.01), 2)}

def store_document(doc_id: str, summary: str) -> dict:
    """Store processed document.
    Args:
        doc_id: Document ID.
        summary: Document summary.
    """
    return {'status': 'stored', 'doc_id': doc_id}

print('5 tools defined')

In [ ]:
# Reusable runner (adapted from Cell 7's test_multi harness).
# Runs one prompt through any agent/workflow and returns the updated session
# so you can inspect session.state (where output_key values land).
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

async def run_agent(agent, prompt, app_name='dm', user_id='u1'):
    ss = InMemorySessionService()
    runner = Runner(agent=agent, app_name=app_name, session_service=ss)
    session = await ss.create_session(app_name=app_name, user_id=user_id)
    print('User: ' + prompt)
    msg = Content(role='user', parts=[Part(text=prompt)])
    async for ev in runner.run_async(
            user_id=user_id, session_id=session.id, new_message=msg):
        if ev.is_final_response() and ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text:
                    print('Agent: ' + p.text[:200])
    return await ss.get_session(
        app_name=app_name, user_id=user_id, session_id=session.id)

print('run_agent helper ready')

## Exercise 1: Two Sub-Agents

**Difficulty:** Easy

Root with SearchAgent and AnalysisAgent. Test routing.

1. Define a SearchAgent and an AnalysisAgent as `LlmAgent`s, each with its own tools.
2. Create a root `LlmAgent` with `sub_agents=[search_agent, analysis_agent]`.
3. Send one search query and one analysis query and confirm the correct sub-agent handles each.

In [ ]:
from google.adk.agents import LlmAgent

search_agent = LlmAgent(
    name='SearchAgent', model='gemini-3.6-flash',
    description='Searches documents by keyword or metadata.',
    instruction='You are the search specialist. Find relevant documents.',
    tools=[search_documents])

analysis_agent = LlmAgent(
    name='AnalysisAgent', model='gemini-3.6-flash',
    description='Analyzes documents: summaries, entities, sentiment.',
    instruction='You are the analysis specialist. Provide thorough analysis.',
    tools=[summarize_document, extract_entities])

root = LlmAgent(
    name='DocuMind', model='gemini-3.6-flash',
    instruction='Route search requests to SearchAgent and analysis requests to AnalysisAgent.',
    sub_agents=[search_agent, analysis_agent])

print('Root agent with 2 sub-agents created')

In [ ]:
# Test routing: each query should land on the matching specialist.
await run_agent(root, 'Find documents about quarterly revenue')
await run_agent(root, 'Summarize document D-01 in detail')

## Exercise 2: SequentialAgent

**Difficulty:** Easy

Two-step pipeline with output_key. Verify state flows.

1. Build an Extractor `LlmAgent` that writes its result to `output_key='extracted_text'`.
2. Build a Classifier `LlmAgent` whose instruction reads `{extracted_text}` from state.
3. Chain both in a `SequentialAgent` and confirm Step 2 consumes Step 1's output.

In [ ]:
from google.adk.agents import SequentialAgent

extractor = LlmAgent(
    name='Extractor', model='gemini-3.6-flash',
    instruction='Extract text and metadata from the document provided.',
    output_key='extracted_text')

classifier = LlmAgent(
    name='Classifier', model='gemini-3.6-flash',
    instruction='Classify this document: {extracted_text}\nDetermine category and sensitivity.',
    output_key='classification')

pipeline = SequentialAgent(
    name='IngestionPipeline',
    sub_agents=[extractor, classifier])

print('Sequential pipeline: Extract -> Classify')

In [ ]:
# Run the pipeline, then inspect state to confirm both keys are populated.
sess = await run_agent(pipeline, 'Document: Acme Q1 revenue rose 12% to $4.2M. Signed by John.')
print('\nState keys:', list(sess.state.keys()))
print('extracted_text present:', 'extracted_text' in sess.state)
print('classification present:', 'classification' in sess.state)

## Exercise 3: Description Routing

**Difficulty:** Easy

Write distinct descriptions. Verify correct sub-agent selected.

1. Add a third specialist, CostAgent, alongside Search and Analysis.
2. Give each sub-agent a sharp, non-overlapping `description` — the root routes on descriptions, not names.
3. Send one query per specialist and confirm each routes correctly.

In [ ]:
# Distinct descriptions are what the root uses to route. Reuse Search/Analysis
# from Exercise 1 and add a pricing specialist.
cost_agent = LlmAgent(
    name='CostAgent', model='gemini-3.6-flash',
    description='Calculates processing costs and pricing for document jobs.',
    instruction='You are the pricing specialist. Calculate costs.',
    tools=[calculate_cost])

router_root = LlmAgent(
    name='DocuMind', model='gemini-3.6-flash',
    instruction='Route search to SearchAgent, analysis to AnalysisAgent, pricing to CostAgent.',
    sub_agents=[search_agent, analysis_agent, cost_agent])

print('Root with 3 distinctly-described sub-agents created')

In [ ]:
await run_agent(router_root, 'Find documents about quarterly revenue')
await run_agent(router_root, 'Extract the named entities from this contract text')
await run_agent(router_root, 'How much would it cost to process 200 pages at premium tier?')

## Exercise 4: Three-Step Pipeline

**Difficulty:** Medium

Extract, Classify, Store with output_key chaining.

1. Reuse the Extract -> Classify pattern from Exercise 2.
2. Add a Storer `LlmAgent` that reads `{classification}` and calls `store_document`, writing to `output_key='storage_result'`.
3. Run the three-stage `SequentialAgent` and confirm state holds `extracted_text`, `classification`, and `storage_result`.

In [ ]:
# Extractor + Classifier lifted from Cell 3; Storer added per the lab hint.
ex_extractor = LlmAgent(
    name='Extractor', model='gemini-3.6-flash',
    instruction='Extract text and metadata from the document provided.',
    output_key='extracted_text')

ex_classifier = LlmAgent(
    name='Classifier', model='gemini-3.6-flash',
    instruction='Classify this document: {extracted_text}\nDetermine category and sensitivity.',
    output_key='classification')

storer = LlmAgent(
    name='Storer', model='gemini-3.6-flash',
    instruction='Persist the processed document. Classification: {classification}. '
                'Call store_document with a doc_id and a one-line summary.',
    tools=[store_document],
    output_key='storage_result')

three_step = SequentialAgent(
    name='IngestStorePipeline',
    sub_agents=[ex_extractor, ex_classifier, storer])

print('Sequential pipeline: Extract -> Classify -> Store')

In [ ]:
sess = await run_agent(three_step, 'Document D-07: Vendor NDA, confidential, signed by Acme legal.')
for key in ['extracted_text', 'classification', 'storage_result']:
    print(key, '->', 'present' if key in sess.state else 'MISSING')

## Exercise 5: ParallelAgent

**Difficulty:** Medium

Two reviewers concurrently + synthesizer.

1. Build Legal and Financial reviewer `LlmAgent`s, each with a unique `output_key`.
2. Run them concurrently inside a `ParallelAgent`.
3. Add a Synthesizer that reads both `output_key`s, and wrap parallel + synth in a `SequentialAgent`.

In [ ]:
from google.adk.agents import ParallelAgent

legal = LlmAgent(name='Legal', model='gemini-3.6-flash',
    instruction='Analyze for legal risks: {doc_text}',
    output_key='legal_review')

financial = LlmAgent(name='Financial', model='gemini-3.6-flash',
    instruction='Analyze financial impact: {doc_text}',
    output_key='financial_review')

parallel = ParallelAgent(
    name='ParallelReview',
    sub_agents=[legal, financial])

synth = LlmAgent(name='Synthesizer', model='gemini-3.6-flash',
    instruction='Combine reviews: Legal={legal_review} Financial={financial_review}',
    output_key='unified_review')

review_workflow = SequentialAgent(
    name='ReviewWorkflow',
    sub_agents=[parallel, synth])

print('Parallel review + synthesis workflow created')

In [ ]:
sess = await run_agent(
    review_workflow,
    'doc_text: Multi-year SaaS contract, auto-renew clause, $500K annual commit.')
for key in ['legal_review', 'financial_review', 'unified_review']:
    print(key, '->', 'present' if key in sess.state else 'MISSING')

## Exercise 6: AgentTool

**Difficulty:** Medium

Wrap SequentialAgent as AgentTool. Invoke from root.

1. Wrap the `IngestionPipeline` (Exercise 2) in an `AgentTool`.
2. Give a root `LlmAgent` that tool so it can invoke the whole workflow as a single callable.
3. Send a query that triggers ingestion and confirm the root returns the pipeline's result.

In [ ]:
from google.adk.tools.agent_tool import AgentTool

# Reuse the pipeline built in Exercise 2.
pipeline_tool = AgentTool(agent=pipeline)

tool_root = LlmAgent(
    name='DocuMind', model='gemini-3.6-flash',
    instruction='When the user hands you a new document to process, call the '
                'IngestionPipeline tool and report what it produced.',
    tools=[pipeline_tool])

print('Root wraps SequentialAgent as an AgentTool')

In [ ]:
await run_agent(tool_root, 'Process this new document: Acme Q2 board minutes, internal only.')

## Exercise 7: LoopAgent

**Difficulty:** Challenge

Critic + Refiner loop with exit_loop and max_iterations=3.

1. Define an `exit_loop` tool that sets `tool_context.actions.escalate = True`.
2. Build a Critic that emits APPROVED-or-feedback and a Refiner that calls `exit_loop` on APPROVED, else improves the draft.
3. Wrap both in a `LoopAgent` with `max_iterations=3` so it stops on approval or after three passes.

In [ ]:
from google.adk.agents import LoopAgent

def exit_loop(tool_context: ToolContext):
    """Call when quality is satisfactory."""
    tool_context.actions.escalate = True
    return {'status': 'Refinement complete'}

critic = LlmAgent(name='Critic', model='gemini-3.6-flash',
    instruction='Review this summary: {draft}. If excellent say APPROVED. Else give fixes.',
    output_key='feedback')

refiner = LlmAgent(name='Refiner', model='gemini-3.6-flash',
    instruction='Feedback: {feedback}. Draft: {draft}. If APPROVED call exit_loop. Else improve.',
    tools=[exit_loop],
    output_key='draft')

refinement = LoopAgent(
    name='RefinementLoop',
    sub_agents=[critic, refiner],
    max_iterations=3)

print('LoopAgent: Critic -> Refiner -> repeat (max 3)')

In [ ]:
# Seed an initial draft in state, then run the loop.
from google.adk.runners import Runner as _Runner
ss = InMemorySessionService()
runner = _Runner(agent=refinement, app_name='dm', session_service=ss)
session = await ss.create_session(
    app_name='dm', user_id='u1', state={'draft': 'acme made money last quarter.'})
msg = Content(role='user', parts=[Part(text='Refine the draft summary until it is excellent.')])
async for ev in runner.run_async(user_id='u1', session_id=session.id, new_message=msg):
    if ev.is_final_response() and ev.content and ev.content.parts:
        for p in ev.content.parts:
            if p.text: print('Agent: ' + p.text[:200])
final = await ss.get_session(app_name='dm', user_id='u1', session_id=session.id)
print('\nFinal draft:', final.state.get('draft'))

## Exercise 8: Full DocuMind

**Difficulty:** Challenge

Complete system: sub_agents + Sequential + Parallel + AgentTool.

1. Combine the three specialists (`sub_agents`) with the ingestion and review workflows exposed as `AgentTool`s.
2. Give the root instructions that separate routing (specialists) from workflow invocation (tools).
3. Run several queries and confirm each path — search, analysis, and pricing — routes correctly.

In [ ]:
# Wrap both workflows as tools and build the full root: 3 sub-agents + 2 workflow tools.
review_tool = AgentTool(agent=review_workflow)

full_root = LlmAgent(
    name='DocuMind', model='gemini-3.6-flash',
    instruction='Route to specialists for queries. Use IngestionPipeline for new docs. '
                'Use ReviewWorkflow for deep analysis.',
    sub_agents=[search_agent, analysis_agent, cost_agent],
    tools=[pipeline_tool, review_tool])

print('Full DocuMind: 3 sub-agents + 2 workflow tools')

In [ ]:
# Exercise the combined system across all routing paths (from Cell 7).
for q in [
    'Find documents about quarterly revenue',
    'Summarize document D-01 in detail',
    'How much would it cost to process 200 pages at premium tier?',
]:
    await run_agent(full_root, q)
    print('---')